In [ ]:
## Importing the libraries
import tqdm, os, time
import torch
import torch.nn as nn
from torch.optim import Adam

from src.datasets_loader import get_loader_segment
from src.score_network_u2ad import ScoreNetwork 
from src.utils import get_lr
from src.evaluation import test
from src.combined_loss import compute_loss_components, compute_final_loss
from src.vpsde import VPSDE
from src.loss_deep_svdd import DSVDDLoss

In [ ]:
## Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Available GPUs: {torch.cuda.device_count()}")
print(f"Current device ID: {torch.cuda.current_device()}")

if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print(f'Allocated: {torch.cuda.memory_allocated(0)/1024**3:.1f} GB')
    print(f'Cached: {torch.cuda.memory_reserved(0)/1024**3:.1f} GB')

### Hyperparameters

In [ ]:
# Define the model parameters
win_size = 100
enc_in = 25
c_out = 25
e_layers = 3
num_steps = 50

# Training parameters
n_epochs = 1
batch_size = 256
lr = 1e-4

# SDE parameters
beta_min = 0.1
beta_max = 20
N = 1000

# Other parameters
anomaly_ratio = 1
k = 3 
eps = 1e-5

In [ ]:
# Declare Dataset
dataset = "PSM"
step = 100

In [ ]:
data_path = './Data-AD/' + dataset + '/'

data_loader = get_loader_segment(
    data_path=data_path,
    batch_size=batch_size,
    win_size=win_size,
    step=step,
    mode='train',
    dataset=dataset)

test_loader = get_loader_segment(
    data_path=data_path,
    batch_size=batch_size,
    win_size=win_size,
    step=step,
    mode='test',
    dataset=dataset)

vali_loader = get_loader_segment(
    data_path=data_path,
    batch_size=batch_size,
    win_size=win_size,
    step=step,
    mode='val',
    dataset=dataset)

thre_loader = get_loader_segment(
    data_path=data_path,
    batch_size=batch_size,
    win_size=win_size,
    step=step,
    mode='thre',
    dataset=dataset)

In [ ]:
## Directory create
saved_model_name = 'u2ad_' + dataset 
saved_model_name_epoch = saved_model_name + '_ckpt_epoch_'

if not os.path.exists(os.path.join('./saved_checkpoints', saved_model_name)):
    os.makedirs(os.path.join('./saved_checkpoints', saved_model_name))
saved_path_str = os.path.join('./saved_checkpoints', saved_model_name, saved_model_name_epoch)

### Model initialize

In [ ]:
sde_model = VPSDE(
    beta_min=beta_min,
    beta_max=beta_max,
    N=N
)

score_model = torch.nn.DataParallel(
    ScoreNetwork(
        marginal_prob_std=sde_model.marginal_prob,
        win_size=win_size,
        enc_in=enc_in,
        c_out=c_out,
        num_steps=num_steps,
        e_layers=e_layers
    )
).to(device)


### Optimizer setup


In [ ]:
optimizer = Adam(score_model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.75)

# Create checkpoint directory
checkpoint_folder = "saved_checkpoints"
if not os.path.exists(checkpoint_folder):
    os.makedirs(checkpoint_folder)

### Training

In [ ]:
## Training

# Initialize SVDD criterion
svdd_criterion = DSVDDLoss(
    net=score_model,
    sde_model=sde_model,
    data_loader=data_loader,
    device=device
)

# Model training
time_now = time.time()
train_steps = len(data_loader)

for epoch in tqdm.tqdm(range(1, n_epochs + 1)):
    curr_lr = get_lr(optimizer)
    print("################### Epoch: ", epoch, " and current learning rate: ", curr_lr, "###################")
    epoch_time = time.time()
    criterion = nn.MSELoss()

    score_model.train()

    for iter_count, (input_data, labels) in enumerate(data_loader):
        score_model.train()
        optimizer.zero_grad()
        input_tensor = input_data.float().to(device)
        
        # Compute loss components 
        loss_sde, loss_rec, loss_global_context, loss_local_context, loss_svdd = compute_loss_components(
            x=input_tensor, 
            sde_model=sde_model, 
            score_model=score_model,
            eps=eps,
            device=device,
            win_size=win_size,
            is_rec_loss=True,
            is_sde_loss=True,
            is_attention_loss=True,
            is_svdd_loss=True,
            svdd_criterion=svdd_criterion,
            ode_solver=True
        )
        
        # Compute final loss 
        loss_combined, loss_global_context_adj, loss_local_context_adj = compute_final_loss(
            loss_sde=loss_sde,
            loss_rec=loss_rec,
            loss_global_context=loss_global_context,
            loss_local_context=loss_local_context,
            loss_svdd=loss_svdd,
            k_1=k,
            k_2=k,
            lamb_k=1,
            normalizing_factor=win_size,
            lamb_k_2=1
        )
        
        # Backward pass
        loss_combined.sum().backward(retain_graph=True)
        loss_local_context_adj.requires_grad = True
        loss_local_context_adj.backward()
        optimizer.step()

    train_loss = torch.mean(loss_combined)
    scheduler.step()

    print("Epoch: {} cost time: {}".format(epoch, time.time() - epoch_time))

### Evaluation

In [ ]:
## EVALUATION
score_model.eval()

with torch.no_grad():
    # Complete evaluation with all metrics
    accuracy, precision, recall, f_score, ADD_value, ADP_value, \
    auroc, auprc, vus_roc, vus_pr, test_energy = test(
        score_model=score_model, 
        anomaly_ratio=anomaly_ratio,
        data_loader=data_loader,
        thre_loader=thre_loader, 
        test_loader=test_loader,
        win_size=win_size,
        sde_model=sde_model,
        eps=eps,
        svdd_criterion=svdd_criterion,
        device=device
    )